# Verifiche strutturali in legno — esempio completo

Verifica SLU e SLE di una **trave principale in legno lamellare GL24h**
secondo NTC18 §4.4.

**Schema**: trave semplicemente appoggiata, luce L = 8 m,
carico permanente + neve (copertura a falda piana).

**Dati materiale** (GL24h — EN 14080):

| Grandezza | Simbolo | Valore |
|-----------|---------|--------|
| Flessione | f_m,k | 24 N/mm² |
| Trazione ‖ | f_t,0,k | 19.2 N/mm² |
| Compressione ‖ | f_c,0,k | 24 N/mm² |
| Taglio | f_v,k | 3.5 N/mm² |
| E medio | E_mean | 11 600 N/mm² |
| E 5° frattile | E_0,05 | 9 400 N/mm² |

**Sezione**: 140 × 400 mm

**Classe di servizio**: 1 (interno) — **Carico dominante**: neve (short_term)

In [1]:
import math
import numpy as np

from pyntc.checks.timber import (
    timber_partial_safety_factor,
    timber_kmod,
    timber_kdef,
    timber_design_strength,
    timber_long_term_modulus,
    timber_km_factor,
    timber_biaxial_bending_check,
    timber_shear_check,
    timber_beam_critical_factor,
    timber_beam_stability_check,
    timber_column_relative_slenderness,
    timber_column_critical_factor,
    timber_column_stability_check,
    timber_deflection_limits,
    timber_straightness_limit,
)
from pyntc.actions.combinations import (
    combination_coefficients,
    slu_combination,
    sle_quasi_permanent_combination,
)

## 1. Materiale — resistenze caratteristiche GL24h

In [2]:
# Resistenze caratteristiche [N/mm²]
f_m_k   = 24.0
f_t_0_k = 19.2
f_c_0_k = 24.0
f_v_k   =  3.5
E_mean  = 11_600.0
E_005   =  9_400.0

# Valori attesi — NTC18 Tab. 4.4.III, produzione standard
expected_gamma_M = 1.45  # GL — produzione standard

gamma_M = timber_partial_safety_factor("glulam", controlled=False)
print(f"gamma_M (GL, standard) = {gamma_M}")

assert math.isclose(gamma_M, expected_gamma_M, rel_tol=1e-3)

gamma_M (GL, standard) = 1.45


## 2. k_mod e k_def — NTC18 Tab. 4.4.IV e 4.4.V

La resistenza di progetto dipende dalla **classe di servizio** e dalla
**durata del carico dominante** attraverso il coefficiente k_mod [4.4.1]:

$$X_d = k_{mod} \cdot \frac{X_k}{\gamma_M}$$

In [3]:
service_class = 1
load_duration = "short_term"

# Valori attesi — NTC18 Tab. 4.4.IV e Tab. 4.4.V
expected_k_mod = 0.90   # GL, classe 1, short_term
expected_k_def = 0.60   # GL, classe 1

k_mod = timber_kmod("glulam", service_class, load_duration)
k_def = timber_kdef("glulam", service_class)

print(f"Classe di servizio : {service_class}")
print(f"Durata carico      : {load_duration}")
print(f"k_mod              = {k_mod}")
print(f"k_def              = {k_def}")

assert math.isclose(k_mod, expected_k_mod, rel_tol=1e-3)
assert math.isclose(k_def, expected_k_def, rel_tol=1e-3)

Classe di servizio : 1
Durata carico      : short_term
k_mod              = 0.9
k_def              = 0.6


## 3. Resistenze di progetto

In [4]:
# Valori attesi — [4.4.1]: X_d = k_mod * X_k / gamma_M
expected_f_m_d   = k_mod * f_m_k   / gamma_M    # = 0.90*24/1.45 = 14.897 N/mm²
expected_f_t_0_d = k_mod * f_t_0_k / gamma_M
expected_f_c_0_d = k_mod * f_c_0_k / gamma_M
expected_f_v_d   = k_mod * f_v_k   / gamma_M    # = 0.90*3.5/1.45 = 2.172  N/mm²
# §4.4.7: E_fin = E_mean / (1 + k_def)
expected_E_fin   = E_mean / (1 + k_def)          # = 11600/1.60 = 7250 N/mm²

f_m_d   = timber_design_strength(f_m_k,   k_mod, gamma_M)
f_t_0_d = timber_design_strength(f_t_0_k, k_mod, gamma_M)
f_c_0_d = timber_design_strength(f_c_0_k, k_mod, gamma_M)
f_v_d   = timber_design_strength(f_v_k,   k_mod, gamma_M)
E_fin   = timber_long_term_modulus(E_mean, k_def)

print(f"f_m,d   = {f_m_d:.3f} N/mm²")
print(f"f_t,0,d = {f_t_0_d:.3f} N/mm²")
print(f"f_c,0,d = {f_c_0_d:.3f} N/mm²")
print(f"f_v,d   = {f_v_d:.3f} N/mm²")
print(f"E_fin   = {E_fin:.0f} N/mm²")

assert math.isclose(f_m_d,   expected_f_m_d,   rel_tol=1e-3)
assert math.isclose(f_v_d,   expected_f_v_d,   rel_tol=1e-3)
assert math.isclose(E_fin,   expected_E_fin,   rel_tol=1e-3)

f_m,d   = 14.897 N/mm²
f_t,0,d = 11.917 N/mm²
f_c,0,d = 14.897 N/mm²
f_v,d   = 2.172 N/mm²
E_fin   = 7250 N/mm²


## 4. Geometria e carichi

In [38]:
# Sezione 200 × 520 mm
b = 260.0    # mm  larghezza
h = 520.0    # mm  altezza
L = 8_000.0  # mm  luce

# Proprieta' sezionali
A  = b * h
I  = b * h**3 / 12
W  = I / (h / 2)

print(f"Sezione  : {b:.0f} × {h:.0f} mm")
print(f"A        = {A:.0f} mm²")
print(f"I        = {I:.3e} mm⁴")
print(f"W        = {W:.0f} mm³")
print()

# Carichi [kN/m]  (trave interasse 5 m, larghezza di influenza)
interasse = 5.0     # m
g1_kNm2   = 0.008 * 11_000 / 1e3  # kN/m²  peso proprio GL24h (gamma = 5 kN/m³)
# Peso proprio trave come carico lineare
G1_lin = 5.0 * b/1000 * h/1000    # kN/m
# Permanente portato (manto + isolante + tavolato)
G2_lin = 1.2 * interasse           # kN/m
# Neve (zona II, altitudine 500 m — da snow_roof_load = 1.06 kN/m²)
Qk_lin = 1.06 * interasse          # kN/m

print(f"G1 (peso proprio trave) = {G1_lin:.3f} kN/m")
print(f"G2 (permanente portato) = {G2_lin:.3f} kN/m")
print(f"Qk (neve)               = {Qk_lin:.3f} kN/m")

Sezione  : 260 × 520 mm
A        = 135200 mm²
I        = 3.047e+09 mm⁴
W        = 11717333 mm³

G1 (peso proprio trave) = 0.676 kN/m
G2 (permanente portato) = 6.000 kN/m
Qk (neve)               = 5.300 kN/m


## 5. Combinazione SLU — NTC18 §2.5.3

In [39]:
psi0_neve, psi1_neve, psi2_neve = combination_coefficients("snow_low")

# Valori attesi — [2.5.1]: γ_G1=1.3, γ_G2=1.5, γ_Q=1.5 (unica variabile: neve)
expected_q_SLU = 1.3 * G1_lin + 1.5 * G2_lin + 1.5 * Qk_lin

q_SLU = slu_combination(
    G1=G1_lin, G2=G2_lin,
    Q=[Qk_lin],
    categories=["snow_low"],
    approach="A1",
)

M_Ed     = q_SLU * (L/1000)**2 / 8
V_Ed     = q_SLU * (L/1000) / 2
M_Ed_Nmm = M_Ed * 1e6
V_Ed_N   = V_Ed * 1e3

print(f"q_SLU = {q_SLU:.3f} kN/m")
print(f"M_Ed  = {M_Ed:.2f} kN·m")
print(f"V_Ed  = {V_Ed:.2f} kN")

assert math.isclose(q_SLU, expected_q_SLU, rel_tol=1e-3)

q_SLU = 17.829 kN/m
M_Ed  = 142.63 kN·m
V_Ed  = 71.32 kN


## 6. Verifica a flessione semplice — NTC18 §4.4.8.1.6, [4.4.4]

$$\frac{\sigma_{m,d}}{f_{m,d}} \leq 1.0$$

In [40]:
# Valori attesi — sigma_m,d = M_Ed/W; η = sigma/f_m,d ≤ 1.0
expected_sigma_m_d = M_Ed_Nmm / W
expected_eta_m     = expected_sigma_m_d / f_m_d

sigma_m_d = M_Ed_Nmm / W
eta_m     = sigma_m_d / f_m_d

print(f"sigma_m,d = {sigma_m_d:.3f} N/mm²")
print(f"f_m,d     = {f_m_d:.3f} N/mm²")
print(f"η = σ/f   = {eta_m:.3f}")
print("✓ Verifica flessione OK" if eta_m <= 1.0 else "✗ NON verificata")

assert math.isclose(sigma_m_d, expected_sigma_m_d, rel_tol=1e-6)
assert math.isclose(eta_m,     expected_eta_m,     rel_tol=1e-6)

sigma_m,d = 12.173 N/mm²
f_m,d     = 14.897 N/mm²
η = σ/f   = 0.817
✓ Verifica flessione OK


## 7. Verifica a taglio — NTC18 §4.4.8.1.9, [4.4.8]

Per sezione rettangolare la tensione tangenziale massima vale
$\tau_d = 1.5 \cdot V_{Ed} / A$.

$$\tau_d \leq f_{v,d}$$

In [41]:
# Valori attesi — [4.4.8]: tau_d = 1.5*V_Ed/A ≤ f_v,d
expected_tau_d = 1.5 * V_Ed_N / A
expected_ratio = expected_tau_d / f_v_d

tau_d = 1.5 * V_Ed_N / A
ok_v, eta_v = timber_shear_check(tau_d, f_v_d)

print(f"tau_d = {tau_d:.3f} N/mm²")
print(f"f_v,d = {f_v_d:.3f} N/mm²")
print(f"η = τ/f_v = {eta_v:.3f}")
print("✓ Verifica taglio OK" if ok_v else "✗ NON verificata")

assert math.isclose(tau_d, expected_tau_d, rel_tol=1e-6)
assert math.isclose(eta_v, expected_ratio, rel_tol=1e-6)

tau_d = 0.791 N/mm²
f_v,d = 2.172 N/mm²
η = τ/f_v = 0.364
✓ Verifica taglio OK


## 8. Verifica di stabilità — svergolamento trave — NTC18 §4.4.8.2.1, [4.4.11]

La trave compressa nella flangia superiore (non vincolata lateralmente) deve
essere verificata allo svergolamento. La tensione critica è (EN 1995-1-1 §6.3.3):

$$\sigma_{m,crit} = \frac{0.78 \, b^2 \, E_{0,05}}{h \, l_{ef}}$$

con $l_{ef} = 0.9 \, L$ per trave appoggiata con carico distribuito.

In [42]:
l_ef = 0.9 * L

# Valori attesi — §6.3.3 EC5 richiamato da NTC18
# sigma_m,crit = 0.78 * b² * E_0,05 / (h * l_ef)
expected_sigma_m_crit = 0.78 * b**2 * E_005 / (h * l_ef)
expected_lambda_rel_m = math.sqrt(f_m_k / expected_sigma_m_crit)
# [4.4.12]: k_crit = 1.0 se λ≤0.75, 1.56-0.75λ se 0.75<λ≤1.4, 1/λ² se λ>1.4
_lr = expected_lambda_rel_m
if _lr <= 0.75:
    expected_k_crit_m = 1.0
elif _lr <= 1.4:
    expected_k_crit_m = 1.56 - 0.75 * _lr
else:
    expected_k_crit_m = 1.0 / _lr**2
expected_eta_stab = sigma_m_d / (expected_k_crit_m * f_m_d)

sigma_m_crit = 0.78 * b**2 * E_005 / (h * l_ef)
lambda_rel_m = math.sqrt(f_m_k / sigma_m_crit)
k_crit_m     = timber_beam_critical_factor(lambda_rel_m)
ok_stab, eta_stab = timber_beam_stability_check(sigma_m_d, f_m_d, lambda_rel_m)

print(f"l_ef           = {l_ef:.0f} mm")
print(f"sigma_m,crit   = {sigma_m_crit:.2f} N/mm²")
print(f"lambda_rel,m   = {lambda_rel_m:.3f}")
print(f"k_crit,m       = {k_crit_m:.3f}")
print(f"η              = {eta_stab:.3f}")
print("✓ Verifica svergolamento OK" if ok_stab else "✗ NON verificata")

assert math.isclose(k_crit_m, expected_k_crit_m, rel_tol=1e-3)
assert math.isclose(eta_stab, expected_eta_stab,  rel_tol=1e-3)

l_ef           = 7200 mm
sigma_m,crit   = 132.38 N/mm²
lambda_rel,m   = 0.426
k_crit,m       = 1.000
η              = 0.817
✓ Verifica svergolamento OK


## 9. Studio parametrico — influenza del vincolo laterale

Aggiungendo vincoli laterali intermedi si riduce l_ef e si migliora k_crit,m.

In [ ]:
n_vincoli_vals = [0, 1, 2, 3, 4]   # vincoli laterali intermedi

print(f"{'n_vincoli':>10}  {'l_ef [m]':>10}  {'λ_rel,m':>10}  "
      f"{'k_crit':>8}  {'η_stab':>8}  {'OK?':>5}")
print("-" * 60)

for n in n_vincoli_vals:
    # Con n vincoli intermedi si formano (n+1) campate
    l_ef_i  = 0.9 * L / (n + 1)
    sc_i    = math.sqrt(f_m_k / (0.78 * b**2 * E_005 / (h * l_ef_i)))
    kc_i    = timber_beam_critical_factor(sc_i)
    _, et_i = timber_beam_stability_check(sigma_m_d, f_m_d, sc_i)
    ok_i    = "✓" if et_i <= 1.0 else "✗"
    print(f"{n:>10d}  {l_ef_i/1000:>10.2f}  {sc_i:>10.3f}  "
          f"{kc_i:>8.3f}  {et_i:>8.3f}  {ok_i:>5}")

 n_vincoli    l_ef [m]     λ_rel,m    k_crit    η_stab    OK?
------------------------------------------------------------
         0        7.20       0.426     1.000     0.817      ✓
         1        3.60       0.301     1.000     0.817      ✓
         2        2.40       0.246     1.000     0.817      ✓
         3        1.80       0.213     1.000     0.817      ✓
         4        1.44       0.190     1.000     0.817      ✓
         5        1.20       0.174     1.000     0.817      ✓
         6        1.03       0.161     1.000     0.817      ✓
         7        0.90       0.151     1.000     0.817      ✓
         8        0.80       0.142     1.000     0.817      ✓
         9        0.72       0.135     1.000     0.817      ✓


## 10. Verifica SLE — deformabilità — NTC18 §4.4.7

**Freccia istantanea** (solo carico variabile):

$$w_{inst} = \frac{5}{384} \cdot \frac{q_k \, L^4}{E_{mean} \, I}$$

**Freccia finale** (azione permanente con E_fin + variabile con k_def):

$$w_{fin} = \frac{5}{384 \, I} \cdot \left[\frac{(G_1+G_2) \, L^4}{E_{fin}} + \frac{Q_k \, L^4}{E_{mean}} \cdot (1 + \psi_{2,neve} \, k_{def})\right]$$

In [44]:
G_lin_Nmm  = (G1_lin + G2_lin) * 1e3 / 1e3
Qk_lin_Nmm = Qk_lin * 1e3 / 1e3
coeff_freccia = 5.0 / 384.0

# Valori attesi — §4.4.7: w ≤ L/300 (istantanea), L/200 (finale)
expected_w_inst     = coeff_freccia * Qk_lin_Nmm * L**4 / (E_mean * I)
expected_w_fin      = coeff_freccia * L**4 / I * (
    G_lin_Nmm / E_fin + Qk_lin_Nmm / E_mean * (1.0 + psi2_neve * k_def)
)
expected_w_lim_inst = L / 300.0   # §4.4.7 trave appoggiata, istantanea
expected_w_lim_fin  = L / 200.0   # §4.4.7 trave appoggiata, finale

w_inst      = coeff_freccia * Qk_lin_Nmm * L**4 / (E_mean * I)
psi2_neve_val = psi2_neve
w_fin = coeff_freccia * L**4 / I * (
    G_lin_Nmm  / E_fin +
    Qk_lin_Nmm / E_mean * (1.0 + psi2_neve_val * k_def)
)
w_lim_inst = timber_deflection_limits(L, "instantaneous")
w_lim_fin  = timber_deflection_limits(L, "final")

print(f"Freccia istantanea  w_inst = {w_inst:.1f} mm   (limite L/300 = {w_lim_inst:.1f} mm)")
print(f"Freccia finale      w_fin  = {w_fin:.1f} mm   (limite L/200 = {w_lim_fin:.1f} mm)")

ok_inst = w_inst <= w_lim_inst
ok_fin  = w_fin  <= w_lim_fin
print("✓ SLE istantanea OK" if ok_inst else "✗ SLE istantanea NON verificata")
print("✓ SLE finale OK"     if ok_fin  else "✗ SLE finale NON verificata")

assert math.isclose(w_inst,     expected_w_inst,     rel_tol=1e-6)
assert math.isclose(w_fin,      expected_w_fin,      rel_tol=1e-6)
assert math.isclose(w_lim_inst, expected_w_lim_inst, rel_tol=1e-6)
assert math.isclose(w_lim_fin,  expected_w_lim_fin,  rel_tol=1e-6)

Freccia istantanea  w_inst = 8.0 mm   (limite L/300 = 26.7 mm)
Freccia finale      w_fin  = 24.1 mm   (limite L/200 = 40.0 mm)
✓ SLE istantanea OK
✓ SLE finale OK


## 11. Studio parametrico — altezza sezione ottimale

Qual è l'altezza minima che soddisfa **tutte** le verifiche?

In [45]:
altezze = np.arange(240, 1020, 20)   # mm

print(f"{'h [mm]':>8}  {'η_M':>6}  {'η_V':>6}  {'η_stab':>8}  {'w_fin/lim':>10}  {'OK?':>5}")
print("-" * 55)

for h_i in altezze:
    A_i  = b * h_i
    I_i  = b * h_i**3 / 12
    W_i  = I_i / (h_i / 2)

    # SLU
    sm_i   = M_Ed_Nmm / W_i
    tau_i  = 1.5 * V_Ed_N / A_i
    eta_M  = sm_i / f_m_d
    eta_V  = tau_i / f_v_d

    # Svergolamento
    lef_i  = 0.9 * L
    sc_i   = math.sqrt(f_m_k / (0.78 * b**2 * E_005 / (h_i * lef_i)))
    _, eta_s = timber_beam_stability_check(sm_i, f_m_d, sc_i)

    # SLE freccia finale
    Ef_i   = timber_long_term_modulus(E_mean, k_def)
    wf_i   = coeff_freccia * L**4 / I_i * (
        G_lin_Nmm  / Ef_i +
        Qk_lin_Nmm / E_mean
    )
    lim_i  = L / 200.0
    ratio_w = wf_i / lim_i

    ok_all = (eta_M <= 1.0 and eta_V <= 1.0 and eta_s <= 1.0 and ratio_w <= 1.0)
    ok_str = "✓" if ok_all else "✗"
    print(f"{h_i:>8.0f}  {eta_M:>6.3f}  {eta_V:>6.3f}  {eta_s:>8.3f}  {ratio_w:>10.3f}  {ok_str:>5}")

  h [mm]     η_M     η_V    η_stab   w_fin/lim    OK?
-------------------------------------------------------
     240   3.836   0.789     3.836       6.133      ✗
     260   3.269   0.728     3.269       4.824      ✗
     280   2.818   0.676     2.818       3.862      ✗
     300   2.455   0.631     2.455       3.140      ✗
     320   2.158   0.592     2.158       2.587      ✗
     340   1.911   0.557     1.911       2.157      ✗
     360   1.705   0.526     1.705       1.817      ✗
     380   1.530   0.498     1.530       1.545      ✗
     400   1.381   0.473     1.381       1.325      ✗
     420   1.253   0.451     1.253       1.144      ✗
     440   1.141   0.430     1.141       0.995      ✗
     460   1.044   0.412     1.044       0.871      ✗
     480   0.959   0.395     0.959       0.767      ✓
     500   0.884   0.379     0.884       0.678      ✓
     520   0.817   0.364     0.817       0.603      ✓
     540   0.758   0.351     0.758       0.538      ✓
     560   0.705   0.338  

## Riepilogo

In [46]:
print("=" * 58)
print("RIEPILOGO VERIFICHE LEGNO LAMELLARE — NTC18 §4.4")
print("=" * 58)
print(f"Materiale      GL24h  k_mod={k_mod}  gamma_M={gamma_M}")
print(f"Sezione        {b:.0f} × {h:.0f} mm    L = {L/1000:.1f} m")
print(f"Classe serv.   {service_class}    Durata carico: {load_duration}")
print("-" * 58)
print(f"Resistenze di progetto:")
print(f"  f_m,d   = {f_m_d:.2f}  N/mm²")
print(f"  f_c,0,d = {f_c_0_d:.2f}  N/mm²")
print(f"  f_v,d   = {f_v_d:.3f} N/mm²")
print("-" * 58)
print(f"Sollecitazioni SLU:")
print(f"  M_Ed = {M_Ed:.2f} kN·m    V_Ed = {V_Ed:.2f} kN")
print("-" * 58)
print(f"SLU Flessione     η = {eta_m:.3f}  {'✓' if eta_m<=1 else '✗'}")
print(f"SLU Taglio        η = {eta_v:.3f}  {'✓' if eta_v<=1 else '✗'}")
print(f"SLU Svergolamento η = {eta_stab:.3f}  {'✓' if eta_stab<=1 else '✗'}  (k_crit={k_crit_m:.3f})")
print("-" * 58)
print(f"SLE w_inst = {w_inst:.1f} mm ≤ {w_lim_inst:.1f} mm  {'✓' if ok_inst else '✗'}")
print(f"SLE w_fin  = {w_fin:.1f} mm ≤ {w_lim_fin:.1f} mm  {'✓' if ok_fin else '✗'}")
print("=" * 58)

RIEPILOGO VERIFICHE LEGNO LAMELLARE — NTC18 §4.4
Materiale      GL24h  k_mod=0.9  gamma_M=1.45
Sezione        260 × 520 mm    L = 8.0 m
Classe serv.   1    Durata carico: short_term
----------------------------------------------------------
Resistenze di progetto:
  f_m,d   = 14.90  N/mm²
  f_c,0,d = 14.90  N/mm²
  f_v,d   = 2.172 N/mm²
----------------------------------------------------------
Sollecitazioni SLU:
  M_Ed = 142.63 kN·m    V_Ed = 71.32 kN
----------------------------------------------------------
SLU Flessione     η = 0.817  ✓
SLU Taglio        η = 0.364  ✓
SLU Svergolamento η = 0.817  ✓  (k_crit=1.000)
----------------------------------------------------------
SLE w_inst = 8.0 mm ≤ 26.7 mm  ✓
SLE w_fin  = 24.1 mm ≤ 40.0 mm  ✓
